# Exploring the BPE tokenizer

Walk through the byte-level BPE pipeline built in `cs336.tokenizer`:
pre-tokenization → training (learning merges) → encode/decode, then
**watch the training loop step by step** and compare the naive vs optimized trainer.

Kernel: **CS336 (.venv)**. If imports fail, pick that kernel (top-right).

In [ ]:
from collections import Counter

from cs336.tokenizer import BPETokenizer, train_bpe, train_bpe_fast
from cs336.tokenizer.bpe import count_pairs, merge_pair, pretokenize

print('imports OK')

## 1. Pre-tokenization

The GPT-2 regex splits text into word-ish chunks *before* BPE runs, so merges never
cross word/punctuation boundaries. Note the leading spaces attached to words.

In [ ]:
pretokenize("Don't merge across words! 123 abc")

## 2. Train BPE and inspect the learned merges

Each merge combines the most frequent adjacent pair (ties → lexicographically greatest).

In [ ]:
corpus = (
    "the cat sat on the mat. "
    "the cat sat. the mat sat. "
    "a cat, a mat, a hat."
) * 20

vocab, merges = train_bpe(corpus, vocab_size=300, special_tokens=["<|endoftext|>"])
print(f"vocab size: {len(vocab)}   num merges: {len(merges)}")
print("\nfirst 15 merges (in learning order):")
for i, (a, b) in enumerate(merges[:15]):
    print(f"  {i:2d}: {a!r} + {b!r} -> {a + b!r}")

## 2b. Watch the merge loop

This is exactly what `train_bpe` does (`cs336/tokenizer/bpe.py:65-74`), reconstructed
from the public primitives so you can *see* every step:

1. **count** every adjacent pair across all words — `count_pairs`
2. **pick** the winner with `max(...)` — most frequent, ties broken by greatest bytes (NOT a sort)
3. **merge** that pair into a new id in every word — `merge_pair`
4. repeat — one new token per round

Each word is shown as its current pieces joined by `·`. Watch `low`, `est`, `new` form.

In [ ]:
text = "low low low lower lower newest newest newest widest widest"

vocab = {i: bytes([i]) for i in range(256)}        # base vocab = 256 raw bytes
words = [[list(w.encode()), c] for w, c in Counter(pretokenize(text)).items()]
next_id = 256

def render(words, vocab):
    # each word = its byte-id pieces decoded for display, joined by middot
    return "   ".join("·".join(vocab[i].decode("latin1") for i in seq) for seq, _ in words)

print("start :", render(words, vocab))
print("        (every character is its own token)\n")

for step in range(8):
    counts = count_pairs([(s, c) for s, c in words])      # 1. count
    # 2. pick winner: max by (freq, greatest-bytes) -- a single pass, NOT a sort
    best = max(counts, key=lambda p: (counts[p], (vocab[p[0]], vocab[p[1]])))
    vocab[next_id] = vocab[best[0]] + vocab[best[1]]
    for w in words:                                       # 3. merge into every word
        w[0] = merge_pair(w[0], best, next_id)
    a, b = vocab[best[0]], vocab[best[1]]
    print(f"merge {step}: {a!r}+{b!r}  (freq {counts[best]})  ->  {vocab[next_id]!r}")
    print("        ", render(words, vocab))
    next_id += 1

## 3. Encode / decode roundtrip

In [ ]:
vocab, merges = train_bpe(corpus, vocab_size=300, special_tokens=["<|endoftext|>"])
tok = BPETokenizer(vocab, merges, special_tokens=["<|endoftext|>"])

text = "the cat sat on the mat"
ids = tok.encode(text)
print("text  :", text)
print("ids   :", ids)
print("tokens:", [vocab[i] for i in ids])
print("decode:", repr(tok.decode(ids)))
print("roundtrip ok:", tok.decode(ids) == text)
print(f"compression: {len(text.encode())} bytes -> {len(ids)} tokens")

## 4. Special tokens stay atomic

`<|endoftext|>` is emitted as a single id and is never split or merged into.

In [ ]:
ids = tok.encode("the cat<|endoftext|>the mat")
eot = next(i for i, b in vocab.items() if b == b"<|endoftext|>")
print("eot id:", eot)
print("ids   :", ids)
print("tokens:", [vocab[i] for i in ids])
print("decode:", repr(tok.decode(ids)))

## 5. Byte-level = handles any Unicode

Even text the tokenizer never trained on roundtrips, because the base vocab is the 256 bytes.

In [ ]:
s = "h\u00e9llo \u4e16\u754c \U0001f680"
print(repr(tok.decode(tok.encode(s))), "==", repr(s), "->", tok.decode(tok.encode(s)) == s)

## 6. Naive vs optimized training

`train_bpe` (naive) recounts **all** pairs and rewrites **all** words every merge → O(merges × corpus).

`train_bpe_fast` keeps pair counts **incrementally** and a `pair → {words}` index, so each
merge only touches the words containing the merged pair. Same result, much faster —
the gap grows with the number of *distinct* words.

In [ ]:
import random
import time

random.seed(1)
# many DISTINCT words -> the naive trainer must rewrite all of them every round
many = ["".join(random.choice("abcdefgh") for _ in range(random.randint(4, 9)))
        for _ in range(6000)]
big_corpus = " ".join(many)
print(f"corpus: {len(big_corpus)} chars, {len(set(many))} distinct words, vocab_size=700\n")

results = {}
for name, fn in [("naive", train_bpe), ("fast", train_bpe_fast)]:
    t = time.perf_counter()
    v, m = fn(big_corpus, 700, [])
    dt = time.perf_counter() - t
    results[name] = (dt, v, m)
    print(f"  {name:5s}: {dt:6.3f}s  ({len(m)} merges)")

slow, fast = results["naive"], results["fast"]
print(f"\n  speedup: {slow[0] / fast[0]:.1f}x")
print("  identical output:", slow[1] == fast[1] and slow[2] == fast[2])